# DataFrames and SQL

This notebook explores the DataFrame API and Spark SQL in depth.

## Learning Objectives

- Master DataFrame operations
- Use Spark SQL for queries
- Understand schemas and data types
- Mix DataFrame API and SQL

In [ ]:
import sys
sys.path.insert(0, "/opt/spark")

from utils.connect_session import get_connect_url
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, max, min, when
# Spark Connect: this notebook is a thin client.
# All computation runs on the Spark cluster (see Spark UI).
spark = (
    SparkSession.builder
    .appName("DataFrames-and-SQL")
    .remote(get_connect_url())   # e.g. sc://spark-master:15002
    .getOrCreate()
)


## 1. Working with Schemas

Schemas define the structure of your data. Understanding them is crucial for data quality.

In [ ]:
from pyspark.sql.types import *

# Define a complex schema
employee_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StructType([
        StructField("first", StringType(), True),
        StructField("last", StringType(), True)
    ]), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("hire_date", DateType(), True),
    StructField("is_active", BooleanType(), True)
])

print("Schema defined with nested structure!")

In [ ]:
# Create sample data
from datetime import date

employees = [
    (1, ("John", "Doe"), "Engineering", 85000.0, date(2020, 1, 15), True),
    (2, ("Jane", "Smith"), "Marketing", 75000.0, date(2019, 6, 1), True),
    (3, ("Bob", "Johnson"), "Engineering", 95000.0, date(2018, 3, 20), True),
    (4, ("Alice", "Williams"), "Sales", 70000.0, date(2021, 9, 10), False),
    (5, ("Charlie", "Brown"), "Engineering", 90000.0, date(2020, 7, 5), True),
]

df = spark.createDataFrame(employees, employee_schema)
df.printSchema()

In [ ]:
# Access nested fields
df.select(
    col("employee_id"),
    col("name.first").alias("first_name"),
    col("name.last").alias("last_name"),
    col("salary")
).show()

## 2. DataFrame API Deep Dive

The DataFrame API provides a rich set of operations for data manipulation.

In [ ]:
# Conditional expressions with when/otherwise
df_with_level = df.withColumn(
    "salary_level",
    when(col("salary") >= 90000, "Senior")
    .when(col("salary") >= 75000, "Mid")
    .otherwise("Junior")
)
df_with_level.select("name.first", "salary", "salary_level").show()

In [ ]:
# Aggregations
df.groupBy("department").agg(
    count("*").alias("employee_count"),
    avg("salary").alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
).show()

In [ ]:
# Filter with multiple conditions
df.filter(
    (col("department") == "Engineering") & 
    (col("salary") > 80000) &
    (col("is_active") == True)
).show()

## 3. Spark SQL

Spark SQL lets you run SQL queries on DataFrames by registering them as temporary views.

In [ ]:
# Register DataFrame as a temporary view
df.createOrReplaceTempView("employees")

# Now we can use SQL!
spark.sql("SELECT * FROM employees WHERE salary > 80000").show()

In [ ]:
# Complex SQL query
query = """
SELECT 
    department,
    COUNT(*) as employee_count,
    ROUND(AVG(salary), 2) as avg_salary
FROM employees
WHERE is_active = true
GROUP BY department
HAVING COUNT(*) > 1
ORDER BY avg_salary DESC
"""

spark.sql(query).show()

In [ ]:
# Access nested fields in SQL
spark.sql("""
SELECT 
    employee_id,
    name.first as first_name,
    name.last as last_name,
    salary
FROM employees
""").show()

## 4. Mixing DataFrame API and SQL

You can freely mix both approaches in your code.

In [ ]:
# Start with DataFrame API
filtered_df = df.filter(col("is_active") == True)

# Register for SQL
filtered_df.createOrReplaceTempView("active_employees")

# Continue with SQL
result = spark.sql("""
SELECT department, AVG(salary) as avg_salary
FROM active_employees
GROUP BY department
""")

# Back to DataFrame API
result.filter(col("avg_salary") > 80000).show()

## 5. Working with Sample Data

Let's work with the generated sample data from the lab.

In [ ]:
# Read the sample data (if generated)
try:
    users = spark.read.parquet("/opt/spark/data/users/small")
    products = spark.read.parquet("/opt/spark/data/products/small")
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    
    print(f"Users: {users.count()} rows")
    print(f"Products: {products.count()} rows")
    print(f"Orders: {orders.count()} rows")
    
    users.printSchema()
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

In [ ]:
# Register views for SQL queries
users.createOrReplaceTempView("users")
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")

# Complex join query
query = """
SELECT 
    u.country,
    COUNT(DISTINCT o.order_id) as order_count,
    SUM(o.total_amount) as total_revenue
FROM users u
JOIN orders o ON u.user_id = o.user_id
GROUP BY u.country
ORDER BY total_revenue DESC
LIMIT 10
"""

spark.sql(query).show()

## 6. Exercises

In [ ]:
# Exercise 1: Find the top 5 products by total sales quantity
# Your code here:


In [ ]:
# Exercise 2: Calculate the average order value per user
# Your code here:


In [ ]:
# Exercise 3: Find users who have never placed an order
# Your code here:


In [ ]:
# Clean up
spark.stop()